# MNIST Generation via Neural Network Reservoir + Linear Readout

Same pipeline as the MoG reservoir notebook, but using a frozen randomly
initialized ReLU network as the reservoir.

Dependencies: `jax`, `optax`, `numpy`, `matplotlib` only.

1. Load MNIST 0s and 1s
2. PCA to low-dimensional space (numpy SVD)
3. Frozen random ReLU network as reservoir
4. Learn linear readout $y = Ax + b$ via MMD
5. Map back to pixel space via inverse PCA

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
from jax import jit, vmap
import optax
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import gzip
import struct
import os

print(f"JAX devices: {jax.devices()}")

## 1. Load MNIST (raw download)

In [ ]:
def download_mnist(data_dir='./mnist_data'):
    os.makedirs(data_dir, exist_ok=True)
    base_url = 'https://storage.googleapis.com/cvdf-datasets/mnist/'
    files = {
        'train_images': 'train-images-idx3-ubyte.gz',
        'train_labels': 'train-labels-idx1-ubyte.gz',
        'test_images':  't10k-images-idx3-ubyte.gz',
        'test_labels':  't10k-labels-idx1-ubyte.gz',
    }
    for key, fname in files.items():
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f'Downloading {fname}...')
            urllib.request.urlretrieve(base_url + fname, fpath)
    
    def read_images(fpath):
        with gzip.open(fpath, 'rb') as f:
            magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
            return np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows * cols)
    
    def read_labels(fpath):
        with gzip.open(fpath, 'rb') as f:
            magic, num = struct.unpack('>II', f.read(8))
            return np.frombuffer(f.read(), dtype=np.uint8)
    
    x_train = read_images(os.path.join(data_dir, files['train_images']))
    y_train = read_labels(os.path.join(data_dir, files['train_labels']))
    x_test  = read_images(os.path.join(data_dir, files['test_images']))
    y_test  = read_labels(os.path.join(data_dir, files['test_labels']))
    x_all = np.concatenate([x_train, x_test]).astype(np.float32) / 255.0
    y_all = np.concatenate([y_train, y_test])
    return x_all, y_all


x_all, y_all = download_mnist()
mask = (y_all == 0) | (y_all == 1)
x_01 = x_all[mask]
y_01 = y_all[mask]

print(f"Total 0s and 1s: {x_01.shape[0]}")
print(f"  0s: {(y_01 == 0).sum()}, 1s: {(y_01 == 1).sum()}")

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(x_01[y_01 == 0][i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(x_01[y_01 == 1][i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('0s'); axes[1, 0].set_ylabel('1s')
plt.suptitle('Training data: 0s and 1s')
plt.tight_layout(); plt.show()

## 2. PCA via numpy SVD

In [ ]:
class SimplePCA:
    def __init__(self, n_components):
        self.n_components = n_components
    
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        X_centered = X - self.mean_
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        self.components_ = Vt[:self.n_components]
        self.singular_values_ = S[:self.n_components]
        total_var = np.sum(S ** 2)
        self.explained_variance_ratio_ = (S[:self.n_components] ** 2) / total_var
        return self
    
    def transform(self, X):
        return (X - self.mean_) @ self.components_.T
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)
    
    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_


N_PCA = 30

pca = SimplePCA(n_components=N_PCA)
x_pca = pca.fit_transform(x_01)

explained = np.cumsum(pca.explained_variance_ratio_)
print(f"PCA to {N_PCA} dims: {explained[-1]*100:.1f}% variance explained")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(1, N_PCA+1), explained, 'o-')
ax.set_xlabel('Number of PCA components')
ax.set_ylabel('Cumulative variance explained')
ax.set_title('PCA of MNIST 0s and 1s')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

x_recon = pca.inverse_transform(x_pca)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(x_01[i].reshape(28, 28), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(x_recon[i].reshape(28, 28), cmap='gray'); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Orig'); axes[1, 0].set_ylabel('Recon')
plt.suptitle(f'PCA reconstruction ({N_PCA} components)')
plt.tight_layout(); plt.show()

print(f"Target data in PCA space: {x_pca.shape}")

## 3. Configuration

In [ ]:
M = N_PCA

# reservoir network
INPUT_DIM = 10       # dimension of Gaussian input to the network
WIDTH = 8*256          # network width W
DEPTH = 1            # network depth D

# training
N_RESERVOIR = 4096
N_TARGET = min(4096, x_pca.shape[0])
LEARNING_RATE = 5e-3
LR_END_RATIO = 1e-2
N_STEPS = 40000
BATCH_SIZE = 512
LOG_EVERY = 1000

# kernel bandwidths via median heuristic
rng_np = np.random.RandomState(0)
subset = x_pca[rng_np.choice(len(x_pca), 500, replace=False)]
diff = subset[:, None, :] - subset[None, :, :]
pw_dists = np.sqrt(np.sum(diff**2, axis=-1))
median_dist = np.median(pw_dists[np.triu_indices(len(subset), k=1)])
print(f"Median pairwise distance in PCA space: {median_dist:.2f}")

SIGMAS = [median_dist * f for f in [0.1, 0.25, 0.5, 1.0, 2.0]]
print(f"Kernel bandwidths: {[f'{s:.2f}' for s in SIGMAS]}")

## 4. Neural Network Reservoir (frozen random ReLU MLP)

In [ ]:
def init_reservoir_nn(key, input_dim, width, depth):
    """Random ReLU MLP: input_dim -> width -> ... -> width (depth layers).
    He initialization. Output is the last hidden layer (dim = width)."""
    params = []
    dims = [input_dim] + [width] * depth
    for i in range(len(dims) - 1):
        key, k1 = jr.split(key)
        scale = jnp.sqrt(2.0 / dims[i])
        W = jr.normal(k1, (dims[i], dims[i + 1])) * scale
        b = jnp.zeros(dims[i + 1])
        params.append((W, b))
    return params


def reservoir_forward_single(params, x):
    """Forward pass for a single input vector."""
    for W, b in params:
        x = jax.nn.relu(x @ W + b)
    return x


reservoir_forward_batch = jit(vmap(reservoir_forward_single, in_axes=(None, 0)))


def sample_reservoir_nn(key, reservoir_params, input_dim, n_samples):
    """Sample Gaussian inputs and pass through the frozen network."""
    inputs = jr.normal(key, (n_samples, input_dim))
    return reservoir_forward_batch(reservoir_params, inputs)


# create reservoir
reservoir_params = init_reservoir_nn(jr.PRNGKey(42), INPUT_DIM, WIDTH, DEPTH)

# test it
test_samples = sample_reservoir_nn(jr.PRNGKey(43), reservoir_params, INPUT_DIM, N_RESERVOIR)
print(f"Reservoir: depth={DEPTH}, width={WIDTH}, input_dim={INPUT_DIM}")
print(f"Reservoir samples: {test_samples.shape}")
print(f"  mean={float(jnp.mean(test_samples)):.4f}, "
      f"std={float(jnp.std(test_samples)):.4f}, "
      f"frac_active={float(jnp.mean(test_samples > 0)):.3f}")

## 5. Target samples

In [ ]:
idx_target = rng_np.choice(len(x_pca), N_TARGET, replace=False)
target_samples = jnp.array(x_pca[idx_target])
target_labels = y_01[idx_target]
print(f"Target samples: {target_samples.shape}")

## 6. MMD loss and readout

In [ ]:
def mmd_squared(P, Q, sigma):
    n, m = P.shape[0], Q.shape[0]
    two_s2 = 2.0 * sigma ** 2
    def sq_dists(X, Y):
        return (jnp.sum(X**2, axis=1, keepdims=True)
                - 2.0 * X @ Y.T
                + jnp.sum(Y**2, axis=1))
    K_pp = jnp.exp(-sq_dists(P, P) / two_s2)
    K_qq = jnp.exp(-sq_dists(Q, Q) / two_s2)
    K_pq = jnp.exp(-sq_dists(P, Q) / two_s2)
    t_pp = (jnp.sum(K_pp) - jnp.trace(K_pp)) / (n * (n - 1))
    t_qq = (jnp.sum(K_qq) - jnp.trace(K_qq)) / (m * (m - 1))
    t_pq = jnp.sum(K_pq) / (n * m)
    return t_pp - 2.0 * t_pq + t_qq


def mmd_multi(P, Q, sigmas):
    return sum(mmd_squared(P, Q, s) for s in sigmas)


def init_readout(key, M, reservoir_width):
    k1, _ = jr.split(key)
    A = jr.normal(k1, (M, reservoir_width)) * 0.01
    b = jnp.zeros(M)
    return {'A': A, 'b': b}


def readout(params, X):
    return X @ params['A'].T + params['b']


def loss_fn(readout_params, res_batch, tgt_batch, sigmas):
    Y = readout(readout_params, res_batch)
    return mmd_multi(Y, tgt_batch, sigmas)

## 7. Training

In [ ]:
readout_params = init_readout(jr.PRNGKey(45), M, WIDTH)

schedule = optax.cosine_decay_schedule(
    init_value=LEARNING_RATE,
    decay_steps=N_STEPS,
    alpha=LR_END_RATIO,
)
optimizer = optax.adam(schedule)
opt_state = optimizer.init(readout_params)


@jit
def train_step(params, opt_state, res_batch, tgt_batch):
    loss, grads = jax.value_and_grad(loss_fn)(params, res_batch, tgt_batch, SIGMAS)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


losses = []
rng = jr.PRNGKey(0)

for step in range(N_STEPS):
    rng, k1, k2 = jr.split(rng, 3)
    # fresh reservoir samples each step
    res_fresh = sample_reservoir_nn(k1, reservoir_params, INPUT_DIM, BATCH_SIZE)
    idx_t = jr.choice(k2, N_TARGET, shape=(BATCH_SIZE,), replace=False)
    
    readout_params, opt_state, loss = train_step(
        readout_params, opt_state,
        res_fresh, target_samples[idx_t]
    )
    losses.append(float(loss))
    
    if step % LOG_EVERY == 0 or step == N_STEPS - 1:
        print(f"step {step:5d}  MMD² = {loss:.6f}")

print("Done.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(losses)
ax.set_xlabel('Step'); ax.set_ylabel('MMD²')
ax.set_title('Training Loss (NN Reservoir)'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Generate and visualize

In [ ]:
N_GEN = 64
gen_reservoir = sample_reservoir_nn(jr.PRNGKey(99), reservoir_params, INPUT_DIM, N_GEN)
gen_pca = readout(readout_params, gen_reservoir)
gen_pixels = np.clip(pca.inverse_transform(np.array(gen_pca)), 0, 1)

fig, axes = plt.subplots(4, 16, figsize=(18, 5))
for i in range(64):
    r, c = i // 16, i % 16
    axes[r, c].imshow(gen_pixels[i].reshape(28, 28), cmap='gray')
    axes[r, c].axis('off')
plt.suptitle('Generated samples (NN reservoir + linear readout)', fontsize=14)
plt.tight_layout()
plt.savefig('mnist_nn_generated.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# side-by-side: real vs generated
fig, axes = plt.subplots(2, 16, figsize=(18, 3))
real_idx = rng_np.choice(len(x_01), 16, replace=False)
for i in range(16):
    axes[0, i].imshow(x_01[real_idx[i]].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(gen_pixels[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Real', fontsize=12)
axes[1, 0].set_ylabel('Gen', fontsize=12)
plt.suptitle('Real vs Generated (NN reservoir)', fontsize=14)
plt.tight_layout()
plt.savefig('mnist_nn_real_vs_gen.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Analysis in PCA space

In [ ]:
gen_pca_large = np.array(readout(readout_params,
    sample_reservoir_nn(jr.PRNGKey(200), reservoir_params, INPUT_DIM, 2048)))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
for digit, color, label in [(0, 'tab:blue', '0'), (1, 'tab:orange', '1')]:
    mask_d = target_labels == digit
    ax.scatter(np.array(target_samples[mask_d, 0]),
              np.array(target_samples[mask_d, 1]),
              alpha=0.2, s=8, c=color, label=label)
ax.set_title('Real (PCA 1-2)'); ax.legend(); ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(gen_pca_large[:, 0], gen_pca_large[:, 1],
          alpha=0.2, s=8, c='tab:red', label='Generated')
ax.set_title('Generated (PCA 1-2)'); ax.legend(); ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.scatter(np.array(target_samples[:, 0]), np.array(target_samples[:, 1]),
          alpha=0.15, s=8, c='tab:blue', label='Real')
ax.scatter(gen_pca_large[:, 0], gen_pca_large[:, 1],
          alpha=0.15, s=8, c='tab:red', label='Generated')
ax.set_title('Overlay (PCA 1-2)'); ax.legend(); ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mnist_nn_pca_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# marginals
n_show = min(6, M)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off'); continue
    ax.hist(np.array(target_samples[:, i]), bins=50, density=True,
            alpha=0.5, color='tab:blue', label='Real')
    ax.hist(gen_pca_large[:, i], bins=50, density=True,
            alpha=0.5, color='tab:red', label='Generated')
    ax.set_title(f'PCA component {i+1}')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Marginal distributions in PCA space (NN reservoir)', fontsize=13)
plt.tight_layout()
plt.savefig('mnist_nn_pca_marginals.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Sweep: Width and Depth

In [ ]:
def run_nn_experiment(width, depth, key, n_steps=5000):
    k1, k2, k3 = jr.split(key, 3)
    rp = init_reservoir_nn(k1, INPUT_DIM, width, depth)
    ro = init_readout(k2, M, width)
    
    sched = optax.cosine_decay_schedule(
        init_value=LEARNING_RATE, decay_steps=n_steps, alpha=LR_END_RATIO
    )
    opt = optax.adam(sched)
    ost = opt.init(ro)
    
    @jit
    def step(ro, ost, rb, tb):
        l, g = jax.value_and_grad(loss_fn)(ro, rb, tb, SIGMAS)
        u, ost = opt.update(g, ost, ro)
        ro = optax.apply_updates(ro, u)
        return ro, ost, l
    
    rng = k3
    tail = []
    for s in range(n_steps):
        rng, rk1, rk2 = jr.split(rng, 3)
        rb = sample_reservoir_nn(rk1, rp, INPUT_DIM, BATCH_SIZE)
        tb = target_samples[jr.choice(rk2, N_TARGET, shape=(BATCH_SIZE,), replace=False)]
        ro, ost, l = step(ro, ost, rb, tb)
        if s >= n_steps - 200:
            tail.append(float(l))
    
    gen = readout(ro, sample_reservoir_nn(jr.PRNGKey(999), rp, INPUT_DIM, 16))
    gen_img = np.clip(pca.inverse_transform(np.array(gen)), 0, 1)
    return np.mean(tail), gen_img


configs = [
    (64,  2),
    (64,  4),
    (128, 2),
    (128, 4),
    (256, 2),
    (256, 4),
    (256, 6),
    (512, 4),
]

sweep_results = {}
sweep_key = jr.PRNGKey(77)

for W_s, D_s in configs:
    sweep_key, sk = jr.split(sweep_key)
    fl, gen_img = run_nn_experiment(W_s, D_s, sk)
    sweep_results[(W_s, D_s)] = (fl, gen_img)
    print(f"W={W_s:4d}  D={D_s}  =>  MMD² = {fl:.6f}")

In [ ]:
n_configs = len(configs)
fig, axes = plt.subplots(n_configs, 16, figsize=(18, 2 * n_configs))

for row, (W_s, D_s) in enumerate(configs):
    fl, gen_img = sweep_results[(W_s, D_s)]
    for col in range(16):
        axes[row, col].imshow(gen_img[col].reshape(28, 28), cmap='gray')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'W={W_s}\nD={D_s}', fontsize=10, rotation=0,
                            labelpad=50, va='center')

plt.suptitle('Generated MNIST across NN reservoir configurations', fontsize=14)
plt.tight_layout()
plt.savefig('mnist_nn_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
labels = [f'W={w}, D={d}' for w, d in configs]
vals = [sweep_results[c][0] for c in configs]
ax.bar(range(len(configs)), vals, tick_label=labels)
ax.set_ylabel('Final MMD²')
ax.set_title('NN Reservoir Sweep Results')
ax.set_yscale('log')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('mnist_nn_sweep_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Diagnostic: reservoir distribution structure

Visualize the reservoir distribution projected to its first 2 PCs.
This helps understand the topological constraint: is the reservoir
distribution a single connected blob, or does it have richer structure?

In [ ]:
# PCA of the reservoir itself
res_large = np.array(sample_reservoir_nn(
    jr.PRNGKey(300), reservoir_params, INPUT_DIM, 4096))

res_pca = SimplePCA(n_components=2)
res_2d = res_pca.fit_transform(res_large)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# reservoir in its own PCA space
axes[0].scatter(res_2d[:, 0], res_2d[:, 1], alpha=0.15, s=5, c='tab:green')
axes[0].set_title(f'Reservoir distribution (PCA)\nW={WIDTH}, D={DEPTH}')
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

# histogram of first PC
axes[1].hist(res_2d[:, 0], bins=80, density=True, alpha=0.7, color='tab:green')
axes[1].set_title('Reservoir PC1 marginal')
axes[1].grid(True, alpha=0.3)

# histogram of norms (to see if it's blobby)
norms = np.linalg.norm(res_large, axis=1)
axes[2].hist(norms, bins=80, density=True, alpha=0.7, color='tab:green')
axes[2].set_title('Reservoir sample norms')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mnist_nn_reservoir_structure.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Reservoir stats:")
print(f"  Fraction of zeros (dead ReLUs): {(res_large == 0).mean():.3f}")
print(f"  Effective dim (variance > 1% of max): "
      f"{np.sum(np.var(res_large, axis=0) > 0.01 * np.max(np.var(res_large, axis=0)))}")